In [0]:
# ============================================================
# STEP 11A: DATA INGESTION CONFIGURATION
# ============================================================

SOURCE_SCHEMA = "workspace.bronze"
TARGET_SCHEMA = "workspace.silver"

SOURCE_TABLES = {
    "companies": f"{SOURCE_SCHEMA}.generated_companies",
    "employees": f"{SOURCE_SCHEMA}.generated_employees_input",
    "ground_truth": f"{SOURCE_SCHEMA}.generated_ground_truth_hierarchy",
    "metadata": f"{SOURCE_SCHEMA}.generated_employee_metadata"
}

TARGET_TABLES = {
    "companies": f"{TARGET_SCHEMA}.companies",
    "employees": f"{TARGET_SCHEMA}.employees",
    "ground_truth": f"{TARGET_SCHEMA}.ground_truth_hierarchy",
    "metadata": f"{TARGET_SCHEMA}.employee_metadata"
}

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")

print("Ingestion configuration initialized.")
print("Source schema:", SOURCE_SCHEMA)
print("Target schema:", TARGET_SCHEMA)

In [0]:
# ============================================================
# STEP 11B: INGEST SYNTHETIC COMPANY AND EMPLOYEE DATA
# ============================================================

# Read generated Bronze tables
companies_bronze = spark.table(
    "workspace.bronze.generated_companies"
)

employees_bronze = spark.table(
    "workspace.bronze.generated_employees_input"
)

# Use separate Silver table names to preserve existing tables
SILVER_COMPANIES = "workspace.silver.generated_companies"
SILVER_EMPLOYEES = "workspace.silver.generated_employees"

# Save companies
(
    companies_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_COMPANIES)
)

# Save inference-facing employee data
(
    employees_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_EMPLOYEES)
)

print("Synthetic company and employee ingestion completed.")

print(
    "Companies ingested:",
    spark.table(SILVER_COMPANIES).count()
)

print(
    "Employees ingested:",
    spark.table(SILVER_EMPLOYEES).count()
)

In [0]:
# ============================================================
# STEP 11C: STORE EVALUATION-ONLY DATASETS
# ============================================================

# Create dedicated evaluation schema
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.evaluation
""")

# Read ground truth and generation metadata from Bronze
ground_truth_bronze = spark.table(
    "workspace.bronze.generated_ground_truth_hierarchy"
)

metadata_bronze = spark.table(
    "workspace.bronze.generated_employee_metadata"
)

# Evaluation-only destination tables
EVALUATION_GROUND_TRUTH = (
    "workspace.evaluation.generated_ground_truth_hierarchy"
)

EVALUATION_METADATA = (
    "workspace.evaluation.generated_employee_metadata"
)

# Save ground truth
(
    ground_truth_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(EVALUATION_GROUND_TRUTH)
)

# Save generation metadata
(
    metadata_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(EVALUATION_METADATA)
)

print("Evaluation datasets stored successfully.")

print(
    "Ground-truth records:",
    spark.table(EVALUATION_GROUND_TRUTH).count()
)

print(
    "Generation metadata records:",
    spark.table(EVALUATION_METADATA).count()
)

In [0]:
# ============================================================
# STEP 11D: VERIFY PROJECT TABLES
# ============================================================

expected_tables = {
    "workspace.bronze.generated_companies": 500,
    "workspace.bronze.generated_employees_input": 74610,
    "workspace.bronze.generated_ground_truth_hierarchy": 74610,
    "workspace.bronze.generated_employee_metadata": 74610,

    "workspace.silver.generated_companies": 500,
    "workspace.silver.generated_employees": 74610,

    "workspace.evaluation.generated_ground_truth_hierarchy": 74610,
    "workspace.evaluation.generated_employee_metadata": 74610
}

for table_name, expected_count in expected_tables.items():

    actual_count = spark.table(table_name).count()

    print(
        f"{table_name} | "
        f"Expected: {expected_count:,} | "
        f"Actual: {actual_count:,}"
    )

    assert actual_count == expected_count, (
        f"Record count mismatch: {table_name}"
    )

print("\nAll expected project tables verified.")